In [1]:
import math


# 大气参数计算相关
# 定义常量
L = -6.5 * (10 ** (-3))  # 温度递减率 (Km^-1)
T0 = 288.15  # 海平面地面温度 (K)
P0 = 101325  # 海平面标准大气压力 (Pa)
M = 0.0289652  # 干空气的摩尔质量 (kg/mol)
R = 8.31446  # 气体常数 (J/(mol·K))
g = 9.80665  # 重力加速度 (m/s^2)


In [29]:

# 失速速度计算相关
def calculate_stall_speed(W, rho, S, C_L_max=1.3):
    """
    此函数用于计算失速速度
    :param W: 飞机重量
    :param rho: 空气密度，会随飞行高度变化
    :param S: 机翼总面积
    :param C_L_max: 最大升力系数，默认值为1.3
    :return: 失速速度
    """
    # 失速速度公式
    V_stall = math.sqrt((2 * W) / (C_L_max * rho * S))
    return V_stall

def calculate_temperature(z):
    """
    此函数用于计算指定高度 z 处的温度
    :param z: 高度，单位为米 (m)
    :return: 高度 z 处的温度，单位为开尔文 (K)
    """
    return T0 - L * z / 1000


def calculate_density(z=0.0, verbose=False):
    """
    此函数用于计算指定高度 z 处的空气密度
    :param z: 高度，单位为米 (m)
    :return: 高度 z 处的空气密度，单位为千克每立方米 (kg/m^3)
    """
    power_z = g * M / (R * L) - 1 
    if verbose:
        print(f"power_z: {power_z}")
    return (P0 * M) / (R * T0) * ((1 - (L * z / T0)) ** power_z)


# 计算爬升/下降功率

def calculate_climb_power(W, V_x, V_y, L_D, eta_mech, eta_prop):
    """
    计算爬升/下降功率
    :param W: 飞机重量
    :param V_x: 水平速度
    :param V_y: 垂直速度
    :param L_D: 爬升/下降阶段的升阻比
    :param eta_mech: 机械效率
    :param eta_prop: 推进效率
    :return: 爬升/下降功率
    """
    return (W * V_y + (W * V_x) / L_D) / (eta_mech * eta_prop)

# Calculate the hover power for open rotor configuration

def calculate_hover_power(f, eta, W, FoM, sigma, rho):
    """
    计算开放式转子配置的悬停功率
    :param f: fuselage correction factor, default is 1.03
    :param eta: 整体机械效率
    :param W: 飞机重量
    :param FoM: figure of merit, referring to the efficiency of rotor power delivery
    :param sigma: 转子载荷
    :param rho: 空气密度
    :return: 悬停功率
    """
    root_sigma = (sigma / (2 * rho)) ** 0.5
    return ((f ** 1.5) * W * root_sigma) / (eta * FoM)

def calculate_cruise_power(W, V, L_D, eta_mech, eta_prop):
    """
    计算巡航功率
    :param W: 飞机重量
    :param V: 水平速度
    :param L_D: 升阻比
    :param eta_mech: 机械效率
    :param eta_prop: 推进效率
    :return: 巡航功率
    """
    return (W * V) / (L_D * eta_mech * eta_prop)

# convert ft to m
def ft_to_m(feet):
    """
    将英尺转换为米
    :param feet: 英尺
    :return: 米
    """
    return feet * 0.3048

def m_to_ft(m):
    """
    将米转换为英尺
    :param m: 米
    :return: 英尺
    """
    return m / 0.3048
# convert lb to kg

In [31]:

from enum import Enum


class FlightPhase(Enum):
    """Docstring for FlightPhase."""
    HOVER = "hover"
    CLIMB = "climb"
    CRUISE = "cruise"
    DESCENT = "descent"
    LANDING = "landing"
    BALKED_ASCENT = "balked_ascent"
    BALKED_CRUISE = "balked_cruise"
    BALKED_DESCENT = "balked_descent"
    BALKED_LANDING = "balked_landing"
    OFFBOARD = "offboard"
  


class Aircraft:
    def __init__(self, name, weight, wing_area, z_curise=1000.0, z_cruise_balked=500.0, flight_range=25.0*1000, t_flight=30.0*60, t_hover_takeoff=30.0, t_hover_landing=60.0, t_balked_landing=60, E_battery=100.0):
        """
        初始化飞机对象
        :param name: 飞机名称
        :param weight: 飞机重量
        :param wing_area: 机翼面积        
        """
        self.name = name
        self.weight = weight
        self.wing_area = wing_area

        self.t = 0 # flight time
        self.t_phase = 0 # phase time since the current phase started
        self.t_phase_all = []

        self.t_flight_all=t_flight # flight time
        self.t_hover_takeoff = t_hover_takeoff
        self.t_hover_landing = t_hover_landing
        self.t_balked_landing = t_balked_landing
        
        self.V_x = 0 # horizontal speed
        self.V_y = 0

        self.x_flight = 0 # horizontal distance
        self.range = flight_range

        self.z_flight = 0 
        self.z_cruise = z_curise
        self.z_cruise_balked = z_cruise_balked
        
        self.P = 0 # power
        self.E_battery = E_battery # battery energy

        self.phase = FlightPhase.HOVER # current phase

    def print_current_state(self):
        """
        打印当前状态
        """
        print(f'x_flight: {self.x_flight:.4f} m; z_flight: {self.z_flight:.4f} m; V_x: {self.V_x:.4f} m/s; V_y: {self.V_y:.4f} m/s')
        print(f'|---phase: {self.phase}; P = {self.P:.4f}; P/E={self.P / self.E_battery:.4f}; t_phase / t_flight (t_max): {self.t_phase:.4f} / {self.t:.4f} ({self.t_flight_all:.4f}) s')

    def change_state(self):
        """
        改变飞机的状态
        :param new_phase: 新的状态
        """
        if self.phase == FlightPhase.HOVER:
            if self.t_phase >= self.t_hover_takeoff:
                # 进入爬升阶段
                self.shift_phase(FlightPhase.CLIMB)

        elif self.phase == FlightPhase.CLIMB:
            if self.z_flight >= self.z_cruise:
                # 进入巡航阶段
                self.shift_phase(FlightPhase.CRUISE)
                
        elif self.phase == FlightPhase.CRUISE:
            # if self.x_flight >= self.range:
            if self.t_phase >= 503.87: # Data found in Ayyaswamy et al. 2023
                # 进入下降阶段        
                self.shift_phase(FlightPhase.DESCENT)        

        elif self.phase == FlightPhase.DESCENT:
            if abs(self.z_flight) <= 0.01:
                # 进入着陆阶段
                self.shift_phase(FlightPhase.LANDING)

        elif self.phase == FlightPhase.LANDING:
            if self.t_phase >= self.t_hover_landing:
                # 进入备降抬升阶段
                self.shift_phase(FlightPhase.BALKED_ASCENT)
        elif self.phase == FlightPhase.BALKED_ASCENT:
            if self.z_flight >= self.z_cruise_balked:
                # 进入巡航阶段
                self.shift_phase(FlightPhase.BALKED_CRUISE)
        elif self.phase == FlightPhase.BALKED_CRUISE:
            if self.t_phase >= 268.6: # Data found in Ayyaswamy et al. 2023
                # 进入下降阶段        
                self.shift_phase(FlightPhase.BALKED_DESCENT)
        elif self.phase == FlightPhase.BALKED_DESCENT:
            if abs(self.z_flight) <= 0.01:
                # 进入着陆阶段
                self.shift_phase(FlightPhase.BALKED_LANDING)
        elif self.phase == FlightPhase.BALKED_LANDING:
            if self.t_phase >= self.t_balked_landing:
                # 进入备降抬升阶段
                self.shift_phase(FlightPhase.OFFBOARD)

    def update_state(self, dt, V_x, V_y, P):
        """
        更新飞机的状态
        :param V_x: 水平速度
        :param V_y: 垂直速度
        :param P: 功率
        """
        self.V_x = V_x
        self.V_y = V_y
        self.P = P
        self.t_phase += dt
        self.t += dt
        self.x_flight += V_x * dt
        self.z_flight += V_y * dt
        self.change_state()


    def shift_phase(self, next_phase):
        """
        更新飞机的状态
        
        """
        # test if phase is a FlightPhase
        if not isinstance(next_phase, FlightPhase):
            raise ValueError(f"Invalid phase: {next_phase}. Valid phase is an instance of class {FlightPhase.name}")
        
        self.print_current_state()
        
        phase_pre = self.phase
        self.phase = next_phase        
        self.t_phase_all.append(self.t_phase)
        self.t_phase = 0 # reset the phase time
        print(f"Phase {phase_pre} changed to: {self.phase} \n")

        


In [4]:
# 示例参数

GTOM = 545.21 # gross takeoff mass (kg)
W = GTOM * g  # 飞机重量（假设值，单位：kg）
sigma = 168.83 # disk loading (kg/m²) = W/S
FoM = 0.7  # figure of merit (FoM), referring to the efficiency of rotor power delivery
f = 1.03  # fuselage correction factor
S = W / sigma  # 机翼面积 (m²)
R_c = 1.3  # 水平速度系数
R_d = 1.0  # 垂直速度系数
L_D_max = 6.0  # 最大升阻比
z_cruise = ft_to_m(1500) # 巡航高度 (m)

mass_battery = 175.21  # 电池质量 (kg)
E_battery = 200 * mass_battery # 电池能量 (Wh)

print(f"mass_battery: {mass_battery:.4f} kg; E_battery: {E_battery:.4f} Wh")

mass_battery: 175.2100 kg; E_battery: 35042.0000 Wh


In [5]:
# calculate the machine efficiency
r_PE = 2.074 # Power to Energy ratio W/Wh

P_hover_true = E_battery * r_PE # 悬停功率 (W)

rho = calculate_density(z=0)

print(f"悬停功率: {P_hover_true:.4f} W; 整机总量= {W:.4f} N; 机翼面积: {S:.4f} m²")

print(f'f**1.5: {f**1.5:.4f}')
print(f'W: {W:.4f} N')
print(f'sigma/ 2 rho ^0.5: {(sigma/ (2*rho))**0.5:.4f}')
root_sigma = (sigma / (2 * rho)) ** 0.5
numerator = (f ** 1.5) * W * root_sigma
denominator = P_hover_true * FoM
print(f'numerator: {numerator:.4f}')
print(f'denominator: {denominator:.4f}')
print(f'root_sigma: {root_sigma:.4f}')

eta_mech = ((f ** 1.5) * W * root_sigma)/ (P_hover_true * FoM) 

print(f"机械效率: {eta_mech:.4f}")

悬停功率: 72677.1080 W; 整机总量= 5346.6836 N; 机翼面积: 31.6690 m²
f**1.5: 1.0453
W: 5346.6836 N
sigma/ 2 rho ^0.5: 8.3012
numerator: 46395.9319
denominator: 50873.9756
root_sigma: 8.3012
机械效率: 0.9120


In [10]:
# determine the efficiency of the propeller

# determine ROC_climb first

t_climb = 120.57 # 爬升时间 (s)

ROC_climb = z_cruise / t_climb 
print(f"ROC_climb = {ROC_climb:.4f} = z_cruise / t_climb: {z_cruise:.4f}/{t_climb:.4f} m")

L_D_climb = ((3 **0.5) / 2) * L_D_max # 4.559  # 爬升阶段的升阻比

print(f'L_D_climb: {L_D_climb:.4f} ( = 3 **0.5 / 2 * L_D_max(= {L_D_max:.4f}))')

rho = calculate_density(z_cruise, True)  # 空气密度 (kg/m³)
V_stall = calculate_stall_speed(W, rho, S)
V_x = V_stall * R_c
V_y = R_d * ROC_climb
print(f'z_cruise: {z_cruise:.4f} m; rho: {rho:.4f} kg/m³')
print(f"V_stall: {V_stall:.4f} m/s; V_x: {V_x:.4f} m/s; V_y: {V_y:.4f} m/s")

P_climb = 1.610 * E_battery # 1.610 is found in Table S2 of Ayyaswamy's paper

eta_prop = (W * V_y + (W * V_x) / L_D_climb) / (eta_mech * P_climb)
print(f"eta_prop: {eta_prop:.4f} ")



ROC_climb = 3.7920 = z_cruise / t_climb: 457.2000/120.5700 m
L_D_climb: 5.1962 ( = 3 **0.5 / 2 * L_D_max(= 6.0000))
power_z: -6.255932779574564
z_cruise: 457.2000 m; rho: 1.1489 kg/m³
V_stall: 15.0361 m/s; V_x: 19.5470 m/s; V_y: 3.7920 m/s
eta_prop: 0.7850 


In [11]:
# calculate the L_D_cruise

r_PE = 1.174 # Power to Energy ratio W/Wh, Found in Table S2 of Ayyaswamy's paper

P_cruise_target = E_battery * r_PE # 巡航功率 (W)
L_D_cruise = (W * V_x) / (P_cruise_target * eta_mech * eta_prop)
print(f"巡航阶段的升阻比: {L_D_cruise:.4f}")

巡航阶段的升阻比: 3.5487


In [25]:
t_descent = 150 # 下降时间 (s)
ROC_descent = -z_cruise / t_descent

print(f"ROC_descent = {ROC_descent:.4f} m/s (= z_cruise / t_descent: {z_cruise:.4f}/{t_descent:.4f})")
rho = calculate_density(z = 0, verbose=False)
V_stall = calculate_stall_speed(W, rho, S)
V_x = V_stall * R_c
V_y = ROC_descent * R_d
r_PE = 0.322
P_descent =  E_battery * r_PE # 下降功率 (W)
L_D_descent = (W * V_x) / (P_descent * eta_mech * eta_prop - W * V_y) 
print(f"下降阶段的升阻比: {L_D_descent:.4f} ( = (W * V_x) / (P_descent * eta_mech * eta_prop - W * V_y))")

ROC_descent = -3.0480 m/s (= z_cruise / t_descent: 457.2000/150.0000)
下降阶段的升阻比: 4.1524 ( = (W * V_x) / (P_descent * eta_mech * eta_prop - W * V_y))


In [30]:
# calculate the theight of balked cruise

z_cruise_balked = ROC_climb * 57.14

print(f"z_cruise_balked: {z_cruise_balked:.4f} m or {m_to_ft(z_cruise_balked)} ft ( = ROC_climb * 57.14)")

z_cruise_balked: 216.6742 m or 710.8733515799951 ft ( = ROC_climb * 57.14)


In [33]:

V_x = 0  # 水平速度 (m/s)
V_y = 0  # 垂直速度 (m/s)

t_all = 30.0 * 60  # seconds, total flight time, 30 mins.
t_hover_takeoff = 30.0  # seconds, hover time in takeoff phase
t_hover_landing = 30.0  # seconds, hover time in landing phase
t_step = 0.01  # seconds, time step
z_cruise = ft_to_m(1500)  # 巡航高度 (m)
range = 25.0 * 1000  # 巡航里程 (m) 25 km

mission = Aircraft(name='open rotor', weight=W, wing_area=S, 
                   z_curise=z_cruise, 
                   z_cruise_balked=z_cruise_balked,
                   flight_range=range, t_flight=t_all, 
                   t_hover_takeoff=t_hover_takeoff, 
                   t_hover_landing=t_hover_landing, 
                   t_balked_landing=60,
                   E_battery=E_battery)

rho = calculate_density()  # 海平面空气密度 (kg/m³)

while mission.t < t_all:
    # calculate the power for the current phase

    if mission.phase == FlightPhase.HOVER:        
        P = calculate_hover_power(f=1.03, eta=eta_mech, W=W, FoM=0.7, sigma=sigma, rho=rho)
    elif mission.phase == FlightPhase.CLIMB:
        rho = calculate_density(z=mission.z_flight)
        V_stall = calculate_stall_speed(W=W, rho=rho, S=S)
        V_x = V_stall * R_c
        V_y = ROC_climb * R_d
        P = calculate_climb_power(W=W, V_x=V_x, V_y=V_y, L_D=L_D_climb, eta_mech=eta_mech, eta_prop=eta_prop)
        
    elif mission.phase == FlightPhase.CRUISE:
        rho = calculate_density(z=mission.z_flight)
        V_stall = calculate_stall_speed(W=W, rho=rho, S=S)
        V_x = V_stall * R_c
        V_y = 0
        P = calculate_cruise_power(W, V=V_x, L_D=L_D_cruise, eta_mech=eta_mech, eta_prop=eta_prop) 
    elif mission.phase == FlightPhase.DESCENT:
        rho = calculate_density(z=mission.z_flight)
        V_stall = calculate_stall_speed(W=W, rho=rho, S=S)
        V_x = V_stall * R_c
        V_y = ROC_descent * R_d
        P = calculate_climb_power(W=W, V_x=V_x, V_y=V_y, L_D=L_D_descent, eta_mech=eta_mech, eta_prop=eta_prop)
        # mission.print_current_state()
    elif mission.phase == FlightPhase.LANDING:
        rho = calculate_density(z=mission.z_flight)
        V_stall = calculate_stall_speed(W=W, rho=rho, S=S)
        V_x = 0.0
        V_y = 0.0
        P = calculate_hover_power(f=1.03, eta=eta_mech, W=W, FoM=0.7, sigma=sigma, rho=rho)     
    elif mission.phase == FlightPhase.BALKED_ASCENT:
        rho = calculate_density(z=mission.z_flight)
        V_stall = calculate_stall_speed(W=W, rho=rho, S=S)
        V_x = V_stall * R_c
        V_y = ROC_climb * R_d
        P = calculate_climb_power(W=W, V_x=V_x, V_y=V_y, L_D=L_D_climb, eta_mech=eta_mech, eta_prop=eta_prop)
    elif mission.phase == FlightPhase.BALKED_CRUISE:
        rho = calculate_density(z=mission.z_cruise_balked)
        V_cruise = R_c * calculate_stall_speed(W=W, rho=rho, S=S)
        # V_x = (1/3 ** 1/4) * V_cruise
        V_x = V_cruise
        V_y = 0
        P = calculate_cruise_power(W, V=V_x, L_D=L_D_cruise, eta_mech=eta_mech, eta_prop=eta_prop)
    elif mission.phase == FlightPhase.BALKED_DESCENT:
        rho = calculate_density(z=mission.z_flight)
        V_stall = calculate_stall_speed(W=W, rho=rho, S=S)
        V_x = V_stall * R_c
        V_y = ROC_descent * R_d
        P = calculate_climb_power(W=W, V_x=V_x, V_y=V_y, L_D=L_D_descent, eta_mech=eta_mech, eta_prop=eta_prop)
    elif mission.phase == FlightPhase.BALKED_LANDING:        
        rho = calculate_density(z=0)
        V_x = 0.0
        V_y = 0.0
        P = calculate_hover_power(f=1.03, eta=eta_mech, W=W, FoM=0.7, sigma=sigma, rho=rho)



    mission.update_state(dt=t_step, V_x=V_x, V_y=V_y, P=P)

print(f"Total time: {mission.t:.2f} s, Total distance: {mission.x_flight:.2f} m, Height: {mission.z_flight:.2f} m")



x_flight: 0.0000 m; z_flight: 0.0000 m; V_x: 0.0000 m/s; V_y: 0.0000 m/s
|---phase: FlightPhase.HOVER; P = 72677.1080; P/E=2.0740; t_phase / t_flight (t_max): 30.0000 / 30.0000 (1800.0000) s
Phase FlightPhase.HOVER changed to: FlightPhase.CLIMB 

x_flight: 2319.6174 m; z_flight: 457.2379 m; V_x: 19.5470 m/s; V_y: 3.7920 m/s
|---phase: FlightPhase.CLIMB; P = 56417.6200; P/E=1.6100; t_phase / t_flight (t_max): 120.5800 / 150.5800 (1800.0000) s
Phase FlightPhase.CLIMB changed to: FlightPhase.CRUISE 

x_flight: 12168.9786 m; z_flight: 457.2379 m; V_x: 19.5470 m/s; V_y: 0.0000 m/s
|---phase: FlightPhase.CRUISE; P = 41139.4169; P/E=1.1740; t_phase / t_flight (t_max): 503.8800 / 654.4600 (1800.0000) s
Phase FlightPhase.CRUISE changed to: FlightPhase.DESCENT 

x_flight: 15054.7534 m; z_flight: 0.0074 m; V_x: 18.9296 m/s; V_y: -3.0480 m/s
|---phase: FlightPhase.DESCENT; P = 11283.6151; P/E=0.3220; t_phase / t_flight (t_max): 150.0100 / 804.4700 (1800.0000) s
Phase FlightPhase.DESCENT changed to